# Data classes

In [77]:
import pandas as pd
import os
import json

templates = {
    tmpl.split(os.sep)[-1] : pd.read_csv(tmpl) if tmpl.endswith(".csv") else json.load(open(tmpl))
    for tmpl in [os.path.join("templates", f) for f in os.listdir("templates")]
}

from dataclasses import dataclass, field

@dataclass
class AbstractTable:
    _field_order : tuple[str] = field(default_factory=tuple)

    @property
    def fields(self):
        return sorted((k for k, v in self.__dataclass_fields__.items() if k != "_field_order"), key=self._field_order.index)
    
    @property
    def data(self):
        return {k : getattr(self, k) for k in self.fields}
    
    def to_row(self, sep=","):
        row_values = []
        for k in self.fields:
            val = str(getattr(self, k) or "")
            # If the value contains a comma or newline, wrap it in quotes
            if sep in val or "\n" in val or '"' in val:
                val = val.replace('"', '""') # escape existing quotes
                val = f'"{val}"'
            row_values.append(val)
        return sep.join(row_values)
    
    def to_header(self):
        return ",".join(self.fields)

    @classmethod
    def validate_schema(cls, template_columns: list[str]):
        """Call this ONCE before parsing your data to ensure columns match."""
        dataclass_fields = set(cls().fields)
        src_columns = set(template_columns)
        
        missing = src_columns - dataclass_fields
        extra = dataclass_fields - src_columns
        
        if missing:
            raise RuntimeError(f'Missing fields in dataclass: {missing}')
        if extra:
            raise RuntimeError(f'Extra fields in dataclass: {extra}')
        
        return True

@dataclass
class Deployment(AbstractTable):
    _field_order : tuple[str] = field(
        default=tuple(templates["deployments.csv"].columns.tolist()),
        init=False,
        repr=False,
        hash=False,
        compare=False
    )
    deploymentID: str | None = None
    locationID: str | None = None
    locationName: str | None = None
    latitude: str | None = None
    longitude: str | None = None
    coordinateUncertainty: str | None = None
    deploymentStart: str | None = None
    deploymentEnd: str | None = None
    setupBy: str | None = None
    cameraID: str | None = None
    cameraModel: str | None = None
    cameraDelay: str | None = None
    cameraHeight: str | None = None
    cameraDepth: str | None = None
    cameraTilt: str | None = None
    cameraHeading: str | None = None
    detectionDistance: str | None = None
    timestampIssues: str | None = None
    baitUse: str | None = None
    featureType: str | None = None
    habitat: str | None = None
    deploymentGroups: str | None = None
    deploymentTags: str | None = None
    deploymentComments: str | None = None
    
@dataclass
class Media(AbstractTable):
    _field_order : tuple[str] = field(
        default=tuple(templates["media.csv"].columns.tolist()),
        init=False,
        repr=False,
        hash=False,
        compare=False
    )
    mediaID: str | None = None
    deploymentID: str | None = None
    captureMethod: str | None = None
    timestamp: str | None = None
    filePath: str | None = None
    filePublic: str | None = None
    fileName: str | None = None
    fileMediatype: str | None = None
    exifData: str | None = None
    favorite: str | None = None
    mediaComments: str | None = None

@dataclass
class Observations(AbstractTable):
    _field_order : tuple[str] = field(
        default=tuple(templates["observations.csv"].columns.tolist()),
        init=False,
        repr=False,
        hash=False,
        compare=False
    )
    observationID: str | None = None
    deploymentID: str | None = None
    mediaID: str | None = None
    eventID: str | None = None
    eventStart: str | None = None
    eventEnd: str | None = None
    observationLevel: str | None = None
    observationType: str | None = None
    cameraSetupType: str | None = None
    scientificName: str | None = None
    count: str | None = None
    lifeStage: str | None = None
    sex: str | None = None
    behavior: str | None = None
    individualID: str | None = None
    individualPositionRadius: str | None = None
    individualPositionAngle: str | None = None
    individualSpeed: str | None = None
    bboxX: str | None = None
    bboxY: str | None = None
    bboxWidth: str | None = None
    bboxHeight: str | None = None
    classificationMethod: str | None = None
    classifiedBy: str | None = None
    classificationTimestamp: str | None = None
    classificationProbability: str | None = None
    observationTags: str | None = None
    observationComments: str | None = None

TABLE_TYPES : dict[str, type[AbstractTable]] = {
    "deployments.csv" : Deployment,
    "media.csv" : Media,
    "observations.csv" : Observations
}

def to_table(type : str, **kwargs):
    cls : type[AbstractTable] = TABLE_TYPES[type]
    return cls(**kwargs)

## Validate data classes

In [78]:
if Deployment.validate_schema(templates["deployments.csv"].columns.tolist()):
    print("Deployment matches template.")
if Media.validate_schema(templates["media.csv"].columns.tolist()):
    print("Media matches template.")
if Observations.validate_schema(templates["observations.csv"].columns.tolist()):
    print("Observations matches template.")

Deployment matches template.
Media matches template.
Observations matches template.


## Conversion abstractions

In [86]:
from dataclasses import dataclass
from typing import Callable, Any, Type
import pandas as pd
import re
from datetime import datetime, timezone

@dataclass
class ColumnMap:
    """Represents a single mapping from a source column to a destination table/column."""
    src_column: str | None
    dst_table: str
    dst_column: str
    # Function signature: takes (column_value, full_row_dict) and returns the processed value
    post_processor: Callable[[Any, dict], Any] = field(default=lambda x, y : x)

class CamtrapCompiler:
    def __init__(self, table_classes: dict[str, Type], mappings: list[ColumnMap]):
        self.table_classes = table_classes
        self.mappings = mappings
        self.raw_data: list[dict] = []
        
        # Used to deduplicate rows so we don't create 5 deployments for 5 observations at one site
        self.primary_keys = {
            "Deployment": "deploymentID",
            "Media": "mediaID",
            "Observations": "observationID"
        }

    def ingest(self, source_data: pd.DataFrame | list[dict]):
        """Loads data into the compiler's memory state."""
        if isinstance(source_data, pd.DataFrame):
            # Convert NaN to None for safer python processing
            self.raw_data = source_data.where(pd.notnull(source_data), None).to_dict('records')
        else:
            self.raw_data = list(source_data)
        print(f"Ingested {len(self.raw_data)} rows.")

    def compile(self, subset: list[str] = None) -> dict[str, list]:
        """
        Compiles the ingested data into table objects. 
        If subset is provided (e.g., ["Observations"]), it will only generate those tables.
        """
        if not self.raw_data:
            raise RuntimeError("No data ingested. Call compile() after ingest().")

        # Determine which tables to build
        target_tables = subset if subset else list(self.table_classes.keys())
        
        # nested dictionary: { "TableName": { "PrimaryKey": ObjectInstance } }
        results = {table: {} for table in target_tables}

        for row in self.raw_data:
            # Temporary bucket for this specific row's compiled data
            row_data = {table: {} for table in target_tables}

            # 1. Evaluate all mappings
            for mapping in self.mappings:
                if mapping.dst_table not in target_tables:
                    continue
                val = mapping.post_processor(row.get(mapping.src_column) if mapping.src_column else None, row)

                # If we have a valid value, assign it as a string
                if val is not None:
                    row_data[mapping.dst_table][mapping.dst_column] = str(val)

            # 2. Instantiate and deduplicate Dataclasses
            for table_name in target_tables:
                pk_field = self.primary_keys[table_name]
                pk_val = row_data[table_name].get(pk_field)

                # Only instantiate if we have a primary key and haven't created it yet
                if pk_val and pk_val not in results[table_name]:
                    cls = self.table_classes[table_name]
                    results[table_name][pk_val] = cls(**row_data[table_name])

        # Flatten the deduplicated dictionaries into lists
        return {table: list(instances.values()) for table, instances in results.items()}
    
# --- Post-Processing Functions ---

def calc_width(max_width : int):
    def inner(val, row) -> float:
        return (float(row['x2']) - float(row['x1'])) / max_width if row.get('x1') and row.get('x2') else None
    return inner

def calc_height(max_height : int):
    def inner(val, row) -> float:
        return (float(row['y2']) - float(row['y1'])) / max_height if row.get('y1') and row.get('y2') else None
    return inner

def parse_longitude(val, row) -> str | None:
    if val:
        match = re.search(r"POINT\(([-\d\.]+)\s([-\d\.]+)\)", str(val))
        return match.group(1) if match else None
    return None

def parse_latitude(val, row) -> str | None:
    if val:
        match = re.search(r"POINT\(([-\d\.]+)\s([-\d\.]+)\)", str(val))
        return match.group(2) if match else None
    return None

def _timestamp_to_iso(x : int | str, timespec="auto"):
    return datetime.fromtimestamp(int(x), timezone.utc).isoformat(timespec=timespec)

def timestamp_to_iso(val, row) -> str | None:
    if val:
        return _timestamp_to_iso(val)
    return None

def hardcode(value : str):
    def inner(val, row) -> str:
        # Ignores the source data and forces a hardcoded value
        return value
    return inner

## Instantiate Converter and Ingest Source Data

In [ ]:
# Source data
df = pd.read_parquet("../raw-data/metadata.parquet")
df = df[df.sourceimageid.notna()] # Source image is required in CamtrapDP

# Create the list of mappings
mappings = [
    # Direct 1-to-1 Mappings
    ColumnMap("detectionid", "Observations", "observationID"),
    ColumnMap("label", "Observations", "scientificName"),
    ColumnMap("score", "Observations", "classificationProbability"),
    ColumnMap("algorithm", "Observations", "classifiedBy"),
    ColumnMap("code", "Deployment", "locationName"),
    ColumnMap("filename", "Media", "fileName"),
    ColumnMap("url", "Media", "filePath"),
    ColumnMap("x1", "Observations", "bboxX", lambda x, y : x / 4096),
    ColumnMap("y1", "Observations", "bboxY", lambda x, y : x / 2160),
    
    # 1-to-Many Splits
    ColumnMap("deploymentid", "Deployment", "deploymentID"),
    ColumnMap("deploymentid", "Media", "deploymentID"),
    ColumnMap("deploymentid", "Observations", "deploymentID"),
    
    ColumnMap("sourceimageid", "Media", "mediaID"),
    ColumnMap("sourceimageid", "Observations", "mediaID"),

    ColumnMap("url", "Media", "filePath"),
    ColumnMap("url", "Media", "fileMediatype", lambda x, y : "image/" + x.split(".")[-1]),

    # Complex / Post-Processed Mappings
    ColumnMap("x2", "Observations", "bboxWidth", calc_width(4096)),
    ColumnMap("y2", "Observations", "bboxHeight", calc_height(2160)),
    ColumnMap("wktposition", "Deployment", "longitude", parse_longitude),
    ColumnMap("wktposition", "Deployment", "latitude", parse_latitude),
    
    ColumnMap("timestamp", "Media", "timestamp", timestamp_to_iso),
    ColumnMap("timestamp", "Observations", "eventStart", timestamp_to_iso),
    ColumnMap("timestamp", "Observations", "eventEnd", timestamp_to_iso),
    
    # Hardcoded Injections (src_column is None)
    ColumnMap(None, "Deployment", "deploymentStart", hardcode(_timestamp_to_iso(df.timestamp.min()))),
    ColumnMap(None, "Deployment", "deploymentEnd", hardcode(_timestamp_to_iso(df.timestamp.max()))),
    ColumnMap(None, "Media", "filePublic", hardcode(True)),
    ColumnMap(None, "Observations", "observationLevel", hardcode("media")),
    ColumnMap(None, "Observations", "observationType", hardcode("animal")),

]

# Initialize Compiler
compiler = CamtrapCompiler(
    table_classes={
        "Deployment": Deployment,
        "Media": Media,
        "Observations": Observations
    },
    mappings=mappings
)

# Ingest Data
compiler.ingest(df)

# 3. Compile everything
all_tables = compiler.compile()


Ingested 11195 rows.


## Convert and Save

In [92]:
import os

def export_to_camtrap_dp(compiled_tables: dict[str, list], output_dir: str):
    """
    Exports the compiled dataclasses to standard Camtrap DP CSV files 
    in the specified directory.
    """
    # Map the class names to standard Camtrap DP filenames
    file_mapping = {
        "Deployment": "deployments.csv",
        "Media": "media.csv",
        "Observations": "observations.csv"
    }
    
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for table_name, instances in compiled_tables.items():
        if not instances:
            print(f"Skipping {table_name}: No data generated.")
            continue
            
        # Get the correct filename, fallback to lowercase class name if not found
        filename = file_mapping.get(table_name, f"{table_name.lower()}.csv")
        filepath = os.path.join(output_dir, filename)
        
        # Using utf-8 encoding is highly recommended for scientific names and special characters
        with open(filepath, "w", encoding="utf-8") as f:
            # Write the header using the first instance
            f.write(instances[0].to_header() + "\n")
            
            # Write all the rows
            for instance in instances:
                f.write(instance.to_row() + "\n")
                
        print(f"Successfully wrote {len(instances)} rows to '{filepath}'")

# 2. Export to a specific directory (e.g., "output/camtrap_export_v1")
export_to_camtrap_dp(all_tables, output_dir="..")

Successfully wrote 1 rows to '../deployments.csv'
Successfully wrote 206 rows to '../media.csv'
Successfully wrote 2239 rows to '../observations.csv'
